# 01 · Preprocesamiento y características

Objetivo: transformar las señales crudas en datos reproducibles, documentar la limpieza y guardar las mismas particiones para todos los modelos.

Proyecto de ECG con dos etapas: reposo/actividad y clasificación de cuatro ritmos. Ejecutar los cuadernos **01 → 02 → 03 → 04** con la misma carpeta de resultados. Estos archivos se entregan sin salidas ejecutadas: no contienen métricas inventadas.


## Preparación en Colab
Instalar dependencias, montar Drive y editar las rutas. Subir previamente los datos a Drive; no hace falta subir el ZIP de 2,5 GB si se usa el Parquet Chapman ya consolidado. Para entrenar ResNet, seleccionar un entorno con GPU. El código también admite CPU, con mayor tiempo de ejecución.

Referencias: [entrada y salida de archivos en Colab](https://colab.research.google.com/notebooks/io.ipynb) y [preguntas frecuentes](https://research.google.com/colaboratory/faq.html).


In [ ]:
%pip -q install pyarrow seaborn joblib scipy scikit-learn skl2onnx onnx


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

# EDITAR: carpeta del proyecto en tu Google Drive. Usar la misma en los 4 cuadernos.
PROJECT_DIR = Path('/content/drive/MyDrive/TP_IA')
REST_DIR = PROJECT_DIR / 'RestingClass' / 'data_resting'
ACTIVE_DIR = PROJECT_DIR / 'RestingClass' / 'data_not_resting'
CHAPMAN_FILE = PROJECT_DIR / 'Detector' / 'chapman_lead_I_consolidated.parquet'
STAGES = [1, 2]  # Usar [2] si primero quieres trabajar solamente con Chapman.
RESULTS = PROJECT_DIR / 'results_colab'
PREPARED = RESULTS / 'prepared'
MODELS = RESULTS / 'models'
FIGURES = RESULTS / 'figures'
for directory in (RESULTS, PREPARED, MODELS, FIGURES):
    directory.mkdir(parents=True, exist_ok=True)
assert STAGES and set(STAGES) <= {1, 2}, 'Elegir etapas 1 y/o 2.'
SEED = 42
print('Resultados:', RESULTS)


In [ ]:
import json, random, uuid, hashlib, gc, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import skew, kurtosis
from scipy.signal import find_peaks, welch
from scipy.interpolate import interp1d
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, StratifiedKFold, GridSearchCV, learning_curve
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, ConfusionMatrixDisplay, classification_report
from IPython.display import display

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASS_NAMES = {1: ['Reposo', 'Actividad'], 2: ['Bradicardia sinusal', 'Taquicardia sinusal', 'Fibrilación auricular', 'Ritmo sinusal normal']}
FEATURE_NAMES = {
    1: ['Kurtosis', 'ZCR', 'Crest_Factor', 'Skewness', 'HR', 'RMSSD', 'SDNN', 'EMG_Power', 'Baseline_Power', 'LF_HF_Ratio', 'Morph_Corr', 'Std_Dev'],
    2: ['hr', 'sdnn', 'rmssd', 'pnn50', 'pnn20', 'rr_cv', 'sd1', 'sd2', 'sd1_sd2', 'skew', 'kurt']
}
def save_json(filename, value):
    Path(filename).write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')

def load_prepared(stage):
    filename = PREPARED / f'stage{stage}.npz'
    if not filename.exists():
        raise FileNotFoundError(f'{filename}: ejecutar primero el cuaderno 01 para esta etapa.')
    with np.load(filename, allow_pickle=False) as stored:
        data = {key: stored[key] for key in stored.files}
    return data

def metrics(y, pred, probs):
    # Macro para comparar con igual peso todas las clases, en ambas etapas.
    auc = roc_auc_score(y, probs[:, 1]) if probs.shape[1] == 2 else roc_auc_score(y, probs, multi_class='ovr', average='macro')
    return dict(accuracy=accuracy_score(y, pred), precision_macro=precision_score(y, pred, average='macro', zero_division=0), recall_macro=recall_score(y, pred, average='macro', zero_division=0), f1_macro=f1_score(y, pred, average='macro', zero_division=0), auc=auc)

def make_cv(stage, y, groups, folds=3):
    cv = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=SEED) if stage == 1 else StratifiedKFold(n_splits=folds, shuffle=True, random_state=SEED)
    splits = list(cv.split(np.zeros((len(y), 1)), y, groups if stage == 1 else None))
    for tr, va in splits:
        if len(np.unique(y[tr])) != len(CLASS_NAMES[stage]) or len(np.unique(y[va])) != len(CLASS_NAMES[stage]):
            raise ValueError('Un fold no contiene todas las clases. Revisar sujetos/clases o reducir folds.')
        if stage == 1:
            assert not set(groups[tr]) & set(groups[va]), 'Sujetos compartidos entre folds.'
    return splits

def make_loader(signals, labels, batch_size, shuffle=False):
    # Normalización por señal; no ajusta estadísticas sobre otras muestras.
    a = np.asarray(signals, dtype=np.float32)
    a = (a - a.mean(axis=1, keepdims=True)) / (a.std(axis=1, keepdims=True) + 1e-8)
    dataset = TensorDataset(torch.from_numpy(a).unsqueeze(1), torch.as_tensor(labels, dtype=torch.long))
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        ce_loss = nn.CrossEntropyLoss(weight=self.alpha, reduction='none')(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


class BasicBlock1D(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(cin, cout, 5, stride=stride, padding=2, bias=False)
        self.bn1 = nn.BatchNorm1d(cout)
        self.conv2 = nn.Conv1d(cout, cout, 5, padding=2, bias=False)
        self.bn2 = nn.BatchNorm1d(cout)
        self.shortcut = nn.Sequential()
        if stride != 1 or cin != cout:
            self.shortcut = nn.Sequential(nn.Conv1d(cin, cout, 1, stride=stride, bias=False), nn.BatchNorm1d(cout))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        return F.relu(self.bn2(self.conv2(out)) + self.shortcut(x))

class ResNet1D(nn.Module):
    def __init__(self, classes, width, dropout):
        super().__init__()
        self.conv1 = nn.Conv1d(1, width, 7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(width)
        self.maxpool = nn.MaxPool1d(3, stride=2, padding=1)
        self.layer1 = nn.Sequential(BasicBlock1D(width, width), BasicBlock1D(width, width))
        self.layer2 = nn.Sequential(BasicBlock1D(width, width*2, 2), BasicBlock1D(width*2, width*2))
        self.layer3 = nn.Sequential(BasicBlock1D(width*2, width*4, 2), BasicBlock1D(width*4, width*4))
        self.layer4 = nn.Sequential(BasicBlock1D(width*4, width*8, 2), BasicBlock1D(width*8, width*8))
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(width*8, classes))
    def forward(self, x):
        x = self.maxpool(F.relu(self.bn1(self.conv1(x))))
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        return self.fc(self.avgpool(x).flatten(1))

def predict_network(model, loader):
    model.eval()
    chunks = []
    with torch.no_grad():
        for signals, _ in loader:
            chunks.append(F.softmax(model(signals.to(DEVICE)), dim=1).cpu().numpy())
    return np.concatenate(chunks)

print('Dispositivo:', DEVICE)
versions = {name: __import__(name).__version__ for name in ['numpy', 'pandas', 'sklearn', 'scipy', 'torch', 'joblib']}
versions['python'] = platform.python_version()
save_json(RESULTS / 'environment.json', versions)
display(versions)


## Datos y características
**Etapa 1:** CSV separados por punto y coma, sin encabezado; se usa la segunda columna como amplitud. Ventanas completas de 4000 muestras (8 s a 500 Hz). El identificador del sujeto se obtiene del prefijo del nombre anterior a _; confirmar que este convenio coincide con tus archivos.

**Etapa 2:** Parquet con columnas record_id, label y signal; se exige exactamente 5000 muestras (10 s a 500 Hz). La frecuencia de 500 Hz se asume según los scripts entregados: verificarla contra la fuente de los datos.

| Característica creada | Generación e interpretación |
|---|---|
| HR | 60 dividido por el promedio de intervalos entre picos R, en latidos/minuto. |
| RMSSD | Raíz del promedio de diferencias sucesivas RR al cuadrado, en segundos. |
| SDNN | Desviación de intervalos RR, en segundos; el código original usa distintas convenciones de ddof entre etapas. |
| ZCR | Proporción de cambios de signo de la señal centrada. |
| Kurtosis / Skewness | Curtosis de Fisher y asimetría de amplitudes. |
| Crest_Factor | Máximo absoluto de la señal centrada y normalizada por su desviación. |
| EMG_Power | Fracción de potencia entre 40 y 150 Hz. |
| Baseline_Power | Fracción de potencia entre 0,1 y 0,5 Hz; Welch con ventana completa da bins cada 0,125 Hz. |
| LF_HF_Ratio | Cociente espectral RR entre bandas LF/HF; estimación limitada por ventanas de solo 8 s. |
| Morph_Corr | Correlación promedio de segmentos de latidos con su plantilla promedio. |
| Std_Dev | Desviación de la amplitud original, con epsilon numérico. |
| pNN50 / pNN20 | Proporción de diferencias RR mayores a 50/20 ms (valores entre 0 y 1). |
| RR_CV | Desviación RR dividida por su media. |
| SD1 / SD2 / SD1_SD2 | Descriptores derivados de la variabilidad RR y cociente entre ellos. |

Se superan las tres características adicionales solicitadas. Si no hay suficientes picos, varios descriptores reciben cero; esto no demuestra ausencia de enfermedad. La detección de picos y las ventanas cortas son limitaciones del método.


In [ ]:
def extract_stage1(window, fs=500):
    mean = np.mean(window)
    std = np.std(window) + 1e-8
    window_norm = (window - mean) / std
    
    f_kurtosis = kurtosis(window_norm)
    f_skew = skew(window_norm)
    zcr = ((window_norm[:-1] * window_norm[1:]) < 0).sum() / len(window)
    crest_factor = np.max(np.abs(window_norm))
    
    freqs, psd = welch(window_norm, fs=fs, nperseg=256)
    total_power = np.sum(psd) + 1e-8
    emg_power = np.sum(psd[(freqs >= 40) & (freqs <= 150)]) / total_power
    # Use the full 8-second window: resolution = 500 / 4000 = 0.125 Hz.
    # The 256-sample EMG spectrum has no bins in the baseline band.
    baseline_freqs, baseline_psd = welch(window_norm, fs=fs, nperseg=len(window_norm))
    baseline_power = np.sum(baseline_psd[(baseline_freqs >= 0.1) & (baseline_freqs <= 0.5)]) / (np.sum(baseline_psd) + 1e-8)
    
    peaks, _ = find_peaks(window, distance=150, prominence=np.std(window) * 0.4)
    if len(peaks) > 2:
        rr = np.diff(peaks) / fs
        hr = 60.0 / np.mean(rr) if np.mean(rr) > 0 else 0.0
        sdnn = np.std(rr, ddof=1)
        rmssd = np.sqrt(np.mean(np.square(np.diff(rr))))
        
        try:
            rr_times = np.cumsum(rr)
            f_interp = interp1d(rr_times, rr, kind='cubic', fill_value="extrapolate")
            t_uniform = np.arange(rr_times[0], rr_times[-1], 0.25)
            rr_uniform = f_interp(t_uniform)
            f_rr, psd_rr = welch(rr_uniform - np.mean(rr_uniform), fs=4.0, nperseg=min(256, len(rr_uniform)))
            lf_power = np.sum(psd_rr[(f_rr >= 0.04) & (f_rr < 0.15)])
            hf_power = np.sum(psd_rr[(f_rr >= 0.15) & (f_rr <= 0.4)])
            lf_hf_ratio = lf_power / (hf_power + 1e-8)
        except:
            lf_hf_ratio = 0.0
            
        beats = []
        for p in peaks:
            if p > 75 and p < len(window) - 75:
                beat = window[p-75:p+75]
                beats.append((beat - np.mean(beat)) / (np.std(beat) + 1e-8))
        
        if len(beats) > 1:
            beats = np.array(beats)
            template = np.mean(beats, axis=0)
            corrs = [np.corrcoef(beat, template)[0, 1] for beat in beats]
            morph_corr = np.nanmean(corrs)
        else:
            morph_corr = 0.0
    else:
        hr, sdnn, rmssd, lf_hf_ratio, morph_corr = 0.0, 0.0, 0.0, 0.0, 0.0

    return np.nan_to_num([
        f_kurtosis, zcr, crest_factor, f_skew, 
        hr, rmssd, sdnn, 
        emg_power, baseline_power, lf_hf_ratio, morph_corr, std
    ])


def extract_stage2(signal, fs=500):
    peaks, _ = find_peaks(signal, distance=150, prominence=0.2)
    if len(peaks) >= 3:
        rr, diff_rr = np.diff(peaks) / fs, np.diff(np.diff(peaks) / fs)
        hr    = 60.0 / np.mean(rr) if np.mean(rr) > 0 else 0.0
        sdnn  = np.std(rr)
        rmssd = np.sqrt(np.mean(diff_rr ** 2)) if len(diff_rr) > 0 else 0.0
        pnn50 = np.mean(np.abs(diff_rr) > 0.05) if len(diff_rr) > 0 else 0.0
        pnn20 = np.mean(np.abs(diff_rr) > 0.02) if len(diff_rr) > 0 else 0.0
        rr_cv = sdnn / np.mean(rr) if np.mean(rr) > 0 else 0.0
        if len(diff_rr) > 0:
            std_diff = np.std(diff_rr)
            sd1    = float(np.sqrt(0.5) * std_diff)
            sd2    = float(np.sqrt(max(2 * sdnn**2 - 0.5 * std_diff**2, 0.0)))
        else:
            sd1 = sd2 = 0.0
        sd1_sd2 = sd1 / sd2 if sd2 > 0 else 0.0
    else:
        hr = sdnn = rmssd = pnn50 = pnn20 = rr_cv = 0.0
        sd1 = sd2 = sd1_sd2 = 0.0

    sig = np.asarray(signal)
    return np.nan_to_num([hr, sdnn, rmssd, pnn50, pnn20, rr_cv, sd1, sd2, sd1_sd2,
                          skew(sig), kurtosis(sig)])


## Comprobación de la corrección espectral
Una señal de prueba con deriva lenta debe tener más potencia basal que una señal de 5 Hz. Esta prueba usa señales artificiales para comprobar el cálculo, no para medir el desempeño del clasificador.


In [ ]:
t = np.arange(4000) / 500
slow = extract_stage1(np.sin(2*np.pi*0.25*t))[8]
fast = extract_stage1(np.sin(2*np.pi*5*t))[8]
assert slow > fast + 0.5, (slow, fast)
assert np.isfinite(extract_stage1(np.ones(4000))).all()
print({'potencia_basal_0.25_Hz': float(slow), 'potencia_basal_5_Hz': float(fast)})


## Limpieza, extracción y guardado
Se registran descartes y errores. En etapa 1 los valores no numéricos se eliminan antes de segmentar, conservando el criterio original; esto puede alterar la continuidad temporal y debe revisarse si hay muchos descartes. No se rellenan señales incompletas. La etapa 1 separa sujetos; la etapa 2 conserva el 80/20 por clase ordenado por identificador del script original, sin afirmar que sea temporal ni independiente por paciente.


In [ ]:
def prepare_stage1():
    signals, labels, subjects, ids, log = [], [], [], [], []
    for folder, label in [(REST_DIR, 0), (ACTIVE_DIR, 1)]:
        if not folder.is_dir():
            raise FileNotFoundError(f'No existe {folder}. Editar REST_DIR/ACTIVE_DIR.')
        files = sorted(folder.glob('*.csv'))
        if not files:
            raise ValueError(f'No hay CSV en {folder}')
        for filename in files:
            item = dict(file=str(filename), label=label, invalid_values=0, accepted_windows=0, rejected_windows=0, trailing_samples=0, error='')
            try:
                frame = pd.read_csv(filename, header=None, sep=';')
                if frame.shape[1] < 2:
                    raise ValueError('Se requieren al menos dos columnas')
                numeric = pd.to_numeric(frame.iloc[:, 1], errors='coerce').to_numpy(dtype=float)
                valid = np.isfinite(numeric)
                item['invalid_values'] = int((~valid).sum())
                signal = numeric[valid]
                item['trailing_samples'] = len(signal) % 4000
                subject = filename.stem.split('_')[0]
                for start in range(0, len(signal)-3999, 4000):
                    window = signal[start:start+4000]
                    if np.std(window) <= 1e-4:
                        item['rejected_windows'] += 1
                        continue
                    signals.append(window.astype(np.float32))
                    labels.append(label)
                    subjects.append(subject)
                    ids.append(f'{label}/{filename.name}:{start//4000}')
                    item['accepted_windows'] += 1
            except Exception as exc:
                item['error'] = str(exc)
            log.append(item)
    pd.DataFrame(log).to_csv(RESULTS / 'stage1_cleaning.csv', index=False)
    if not signals:
        raise ValueError('Sin ventanas válidas; revisar stage1_cleaning.csv')
    return np.stack(signals), np.asarray(labels), np.asarray(subjects), np.asarray(ids), extract_stage1

def prepare_stage2():
    if not CHAPMAN_FILE.is_file():
        raise FileNotFoundError(f'No existe {CHAPMAN_FILE}. Subir el Parquet consolidado a Drive.')
    frame = pd.read_parquet(CHAPMAN_FILE)
    assert {'record_id', 'label', 'signal'} <= set(frame.columns), 'Esquema Parquet incorrecto'
    label_map = {'Sinus Bradycardia': 0, 'Sinus Tachycardia': 1, 'Atrial Fibrillation': 2, 'Normal Sinus Rhythm': 3}
    signals, labels, ids, log = [], [], [], []
    frame['record_id'] = frame['record_id'].astype(str)
    frame = frame.sort_values('record_id').reset_index(drop=True)
    if frame['record_id'].duplicated().any():
        raise ValueError('record_id duplicados: revisar antes de separar los datos.')
    for row in frame.itertuples(index=False):
        error = ''
        try:
            signal = np.asarray(row.signal, dtype=np.float32)
            if row.label not in label_map:
                raise ValueError('Etiqueta fuera de las cuatro clases')
            if signal.ndim != 1 or len(signal) != 5000:
                raise ValueError('Longitud distinta de 5000')
            if not np.isfinite(signal).all() or np.std(signal) <= 1e-8:
                raise ValueError('Señal no finita o constante')
            signals.append(signal)
            labels.append(label_map[row.label])
            ids.append(row.record_id)
        except Exception as exc:
            error = str(exc)
        log.append(dict(record_id=row.record_id, error=error, accepted=not bool(error)))
    pd.DataFrame(log).to_csv(RESULTS / 'stage2_cleaning.csv', index=False)
    if not signals:
        raise ValueError('No quedaron registros válidos; revisar stage2_cleaning.csv')
    return np.stack(signals), np.asarray(labels), np.asarray(ids), np.asarray(ids), extract_stage2

for stage in STAGES:
    raw, y, groups, ids, extractor = prepare_stage1() if stage == 1 else prepare_stage2()
    features = np.asarray([extractor(signal) for signal in raw], dtype=np.float32)
    assert np.isfinite(features).all(), 'Hay características no finitas'
    if stage == 1:
        tr, va = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED).split(features, y, groups))
        assert not set(groups[tr]) & set(groups[va]), 'Sujetos compartidos'
    else:
        tr, va = [], []
        for label in range(4):
            indices = np.flatnonzero(y == label)
            cut = int(len(indices)*0.8)
            tr.extend(indices[:cut]); va.extend(indices[cut:])
        tr, va = np.asarray(tr, dtype=int), np.asarray(va, dtype=int)
    assert len(np.unique(y[tr])) == len(CLASS_NAMES[stage])
    assert len(np.unique(y[va])) == len(CLASS_NAMES[stage]), 'Faltan clases en validación'
    assert not set(ids[tr]) & set(ids[va])
    make_cv(stage, y[tr], groups[tr])
    dataset_id = str(uuid.uuid4())
    np.savez_compressed(PREPARED / f'stage{stage}.npz', raw=raw, features=features, y=y, groups=groups, ids=ids, train_idx=tr, val_idx=va, dataset_id=np.asarray(dataset_id))
    splits = pd.DataFrame({'id': ids, 'group': groups, 'label': y, 'partition': 'train'})
    splits.loc[va, 'partition'] = 'validation'
    splits.to_csv(RESULTS / f'stage{stage}_partitions.csv', index=False)
    table = pd.DataFrame(features, columns=FEATURE_NAMES[stage])
    table.insert(0, 'id', ids)
    table['label'] = y
    table.to_csv(RESULTS / f'stage{stage}_features.csv', index=False)
    save_json(PREPARED / f'stage{stage}_manifest.json', dict(dataset_id=dataset_id, samples=len(y), train=len(tr), validation=len(va), features=FEATURE_NAMES[stage], classes=CLASS_NAMES[stage], seed=SEED, sampling_rate_assumed=500))
    print(f'Etapa {stage}: {len(tr)} entrenamiento / {len(va)} validación')
    display(pd.crosstab(splits['partition'], splits['label']))
    display(table[FEATURE_NAMES[stage]].describe().T)
    stds = features.std(axis=0)
    print('Características constantes:', [name for name, sd in zip(FEATURE_NAMES[stage], stds) if sd < 1e-8])
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(np.arange(raw.shape[1])/500, raw[0])
    ax.set(xlabel='Tiempo (s)', ylabel='Amplitud (unidades del archivo)', title=f'Etapa {stage}: ejemplo real, {CLASS_NAMES[stage][y[0]]}')
    fig.tight_layout(); fig.savefig(FIGURES / f'stage{stage}_signal_example.png'); plt.show(); plt.close(fig)
    del raw, features, table
    gc.collect()


## Para explicar en la presentación
Registrar cuántos archivos/señales se descartaron, cantidad de sujetos y registros por clase, tres características con ejemplos y la política de partición. Si se vuelve a ejecutar este cuaderno, repetir 02 y 03: se crea una nueva versión de los datos y los modelos anteriores se consideran desactualizados.
